# Context Window Feasibility Analysis

## Overview

Following dataset preprocessing and prompt construction, the next stage of the proposed framework is to determine whether the generated prompts can be accommodated within the context windows of the selected Large Language Models (LLMs).

The proposed framework evaluates six prompt configurations formed by combining three prompting strategies (Zero-shot, Zero-shot Chain-of-Thought, and Few-shot Chain-of-Thought) with two input representations (Code Only and Code + Context).

This notebook generates the actual prompts using the developed prompt builder and measures their token usage using the tokenizer of each selected LLM. The analysis determines whether the generated prompts can be directly used for evaluation or whether additional context reduction is required.

---

## Objectives

This notebook aims to:

- Validate the preprocessed dataset.
- Generate prompts for all planned experimental configurations.
- Measure prompt lengths using the tokenizer of each selected LLM.
- Compare prompt sizes against the context windows of the selected models.
- Determine whether additional context reduction is required before evaluation.

---

## Experimental Configurations

| Prompt Strategy | Code Only | Code + Context |
|-----------------|-----------|----------------|
| Zero-shot | ✓ | ✓ |
| Zero-shot CoT | ✓ | ✓ |
| Few-shot CoT | ✓ | ✓ |

This results in six experimental configurations that will be evaluated throughout the remainder of this study.

In [1]:
# ============================================================
# Step 1 - Import Libraries
# ============================================================

import sys
from pathlib import Path

import pandas as pd
import numpy as np

from IPython.display import display

In [2]:
# ============================================================
# Step 2 - Load Preprocessed Dataset
# ============================================================

PROJECT_ROOT = Path.cwd().parent.parent

DATASET_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "evaluation_dataset.jsonl"
)

working_df = pd.read_json(
    DATASET_PATH,
    lines=True
)

print("=" * 80)
print("DATASET LOADED")
print("=" * 80)

print(f"Dataset Shape : {working_df.shape}")

DATASET LOADED
Dataset Shape : (2741, 15)


## Step 3 - Experimental Configurations

The feasibility analysis is performed using the same prompt configurations that will be used during the final LLM evaluation.

Three prompting strategies are considered:

- Zero-shot
- Zero-shot Chain-of-Thought (CoT)
- Few-shot Chain-of-Thought (CoT)

Each strategy is evaluated under two input representations:

- Code Only
- Code + Context

This results in six experimental configurations.

In [3]:
# ============================================================
# Step 3 - Experimental Configurations
# ============================================================

PROMPT_CONFIGS = [

    {
        "name": "Zero-shot + Code Only",
        "strategy": "zero_shot",
        "include_context": False
    },

    {
        "name": "Zero-shot + Code + Context",
        "strategy": "zero_shot",
        "include_context": True
    },

    {
        "name": "Zero-shot CoT + Code Only",
        "strategy": "zero_shot_cot",
        "include_context": False
    },

    {
        "name": "Zero-shot CoT + Code + Context",
        "strategy": "zero_shot_cot",
        "include_context": True
    },

    {
        "name": "Few-shot CoT + Code Only",
        "strategy": "few_shot_cot",
        "include_context": False
    },

    {
        "name": "Few-shot CoT + Code + Context",
        "strategy": "few_shot_cot",
        "include_context": True
    }

]

print("=" * 80)
print("EXPERIMENTAL CONFIGURATIONS")
print("=" * 80)

display(pd.DataFrame(PROMPT_CONFIGS))

EXPERIMENTAL CONFIGURATIONS


,name,strategy,include_context
0,Zero-shot + Code Only,zero_shot,False
1,Zero-shot + Code + Context,zero_shot,True
2,Zero-shot CoT + Code Only,zero_shot_cot,False
3,Zero-shot CoT + Code + Context,zero_shot_cot,True
4,Few-shot CoT + Code Only,few_shot_cot,False
5,Few-shot CoT + Code + Context,few_shot_cot,True


## Step 4 - Load Prompt Builder

The prompts are generated using the prompt builder developed during the previous stage. Using the same prompt builder for both feasibility analysis and the final evaluation ensures methodological consistency throughout the study.

In [4]:
# ============================================================
# Step 4 - Load Prompt Builder
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.prompt_builder import build_prompt

print("=" * 80)
print("PROMPT BUILDER")
print("=" * 80)

print("✓ Prompt Builder Loaded Successfully")

PROMPT BUILDER
✓ Prompt Builder Loaded Successfully


## Step 5 - Verify Prompt Generation

Before analyzing all test cases, a representative sample is used to verify that prompts are generated correctly for each experimental configuration.

This step confirms that:

- The correct prompting strategy is applied.
- Contextual artifacts are included only when required.
- The generated prompts match the format intended for the final evaluation.

In [5]:
# ============================================================
# Step 5 - Verify Prompt Generation
# ============================================================

sample = working_df.iloc[0].to_dict()

for config in PROMPT_CONFIGS:

    print("=" * 80)
    print(config["name"])
    print("=" * 80)

    prompt = build_prompt(

        sample=sample,

        strategy=config["strategy"],

        include_context=config["include_context"]

    )

    print(prompt[:10500])
    print("\n")

Zero-shot + Code Only
You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the execution order of te

## Step 6 - Load Prompt Tokenizer

To estimate the size of each generated prompt, a single representative tokenizer (Qwen3-Coder) is used to calculate the number of tokens in each prompt.

The estimated token counts are then compared against the published context window sizes of the selected LLMs (Qwen3-Coder, DeepSeek-R1, and Llama-4 Maverick) to verify that all prompt configurations fit within the supported context limits.

## Step 7 - Load the Prompt Tokenizer

The Qwen3-Coder tokenizer is loaded to estimate the number of tokens in each generated prompt. The tokenizer is used only for prompt size estimation and not for model inference.

In [6]:
# ============================================================
# Step 7 - Load Prompt Tokenizer
# ============================================================

from transformers import AutoTokenizer

TOKENIZER_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

print(f"Loading tokenizer: {TOKENIZER_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME,
    trust_remote_code=True
)

print("✓ Tokenizer loaded successfully!")

C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loading tokenizer: Qwen/Qwen3-Coder-30B-A3B-Instruct


✓ Tokenizer loaded successfully!


## Step 8 - Define Context Window Sizes

The published context window sizes of the selected LLMs are defined. These values are later used to determine whether each generated prompt fits within the supported input length of each model.

In [7]:
# ============================================================
# Step 8 - Selected LLM Context Windows
# ============================================================

MODELS = [
    {
        "name": "Qwen3-Coder",
        "context_window": 32768
    },
    {
        "name": "DeepSeek-R1",
        "context_window": 32768
    },
    {
        "name": "Llama-4-Maverick",
        "context_window": 131072
    }
]

MODELS

[{'name': 'Qwen3-Coder', 'context_window': 32768},
 {'name': 'DeepSeek-R1', 'context_window': 32768},
 {'name': 'Llama-4-Maverick', 'context_window': 131072}]

## Step 9 - Generate Prompt Token Statistics

Each dataset instance is processed using every prompt configuration. The generated prompts are tokenized using the Qwen3-Coder tokenizer to estimate their input size.

For each generated prompt, the following information is recorded:

- Prompt configuration
- Character count
- Token count
- Context window utilization
- Whether the prompt fits within each selected LLM's context window

The analysis is performed across the entire dataset to determine the maximum prompt size and verify compatibility with the selected LLMs.

In [8]:
# ============================================================
# Step 9 - Generate Prompt Token Statistics
# ============================================================

import pandas as pd
from tqdm.auto import tqdm

results = []

total_prompts = len(working_df) * len(PROMPT_CONFIGS)

print(f"Dataset samples : {len(working_df):,}")
print(f"Prompt strategies : {len(PROMPT_CONFIGS)}")
print(f"Total prompts : {total_prompts:,}")
print()

for sample_index, (_, row) in enumerate(tqdm(working_df.iterrows(),
                                              total=len(working_df),
                                              desc="Generating Prompts")):

    for config in PROMPT_CONFIGS:

        # Build prompt
        prompt = build_prompt(
            sample=row,
            strategy=config["strategy"],
            include_context=config["include_context"]
        )

        # Prompt statistics
        char_count = len(prompt)
        token_count = len(tokenizer.encode(prompt))

        # Compare against every selected LLM
        for model in MODELS:

            utilization = (token_count / model["context_window"]) * 100

            results.append({
                "sample_index": sample_index,
                "prompt_configuration": config["name"],
                "strategy": config["strategy"],
                "include_context": config["include_context"],

                "character_count": char_count,
                "token_count": token_count,

                "model": model["name"],
                "context_window": model["context_window"],

                "utilization_percent": round(utilization, 2),
                "fits_context": token_count <= model["context_window"]
            })

results_df = pd.DataFrame(results)

print()
print("Analysis completed successfully.")
print(f"Generated records : {len(results_df):,}")

Dataset samples : 2,741
Prompt strategies : 6
Total prompts : 16,446



Generating Prompts: 100%|██████████| 2741/2741 [02:50<00:00, 16.05it/s]



Analysis completed successfully.
Generated records : 49,338


## Step 10 - Summarize Prompt Statistics

Descriptive statistics are calculated for each prompt configuration to analyze the distribution of prompt sizes across the dataset.

The statistics include:

- Minimum token count
- Average token count
- Median token count
- 95th percentile
- 99th percentile
- Maximum token count

These values are used to identify the largest generated prompts and verify that they remain within the supported context windows of the selected LLMs.

In [9]:
# ============================================================
# Step 10 - Prompt Token Statistics
# ============================================================

summary_df = (
    results_df
    .drop_duplicates(
        subset=["sample_index", "prompt_configuration"]
    )
    .groupby("prompt_configuration")["token_count"]
    .agg(
        Minimum="min",
        Mean="mean",
        Median="median",
        Percentile95=lambda x: x.quantile(0.95),
        Percentile99=lambda x: x.quantile(0.99),
        Maximum="max"
    )
    .round(2)
    .reset_index()
)

summary_df

,prompt_configuration,Minimum,Mean,Median,Percentile95,Percentile99,Maximum
0,Few-shot CoT + Code + Context,3561,9364.43,5259.0,24281.0,62882.8,237416
1,Few-shot CoT + Code Only,1192,1349.83,1288.0,1648.0,2182.4,9527
2,Zero-shot + Code + Context,503,6306.43,2201.0,21223.0,59824.8,234358
3,Zero-shot + Code Only,399,556.83,495.0,855.0,1389.4,8734
4,Zero-shot CoT + Code + Context,537,6340.43,2235.0,21257.0,59858.8,234392
5,Zero-shot CoT + Code Only,433,590.83,529.0,889.0,1423.4,8768


## Step 11 - Count Prompts Exceeding Context Windows

The number of generated prompts that exceed the maximum context window of each selected LLM is calculated. This analysis identifies whether any prompt configurations require preprocessing or context reduction before being submitted to the models.

In [10]:
# ============================================================
# Step 11 - Count Prompts Exceeding Context Windows
# ============================================================

context_window_summary = (
    results_df
    .groupby(["model", "prompt_configuration"])
    .agg(
        Total_Prompts=("fits_context", "count"),
        Exceeds_Context=("fits_context", lambda x: (~x).sum()),
        Fits_Context=("fits_context", "sum")
    )
    .reset_index()
)

context_window_summary["Exceeds (%)"] = (
    context_window_summary["Exceeds_Context"]
    / context_window_summary["Total_Prompts"] * 100
).round(2)

context_window_summary["Fits (%)"] = (
    context_window_summary["Fits_Context"]
    / context_window_summary["Total_Prompts"] * 100
).round(2)

context_window_summary

,model,prompt_configuration,Total_Prompts,Exceeds_Context,Fits_Context,Exceeds (%),Fits (%)
0,DeepSeek-R1,Few-shot CoT + Code + Context,2741,63,2678,2.30,97.70
1,DeepSeek-R1,Few-shot CoT + Code Only,2741,0,2741,0.00,100.00
2,DeepSeek-R1,Zero-shot + Code + Context,2741,60,2681,2.19,97.81
3,DeepSeek-R1,Zero-shot + Code Only,2741,0,2741,0.00,100.00
4,DeepSeek-R1,Zero-shot CoT + Code + Context,2741,60,2681,2.19,97.81
5,DeepSeek-R1,Zero-shot CoT + Code Only,2741,0,2741,0.00,100.00
6,Llama-4-Maverick,Few-shot CoT + Code + Context,2741,2,2739,0.07,99.93
7,Llama-4-Maverick,Few-shot CoT + Code Only,2741,0,2741,0.00,100.00
8,Llama-4-Maverick,Zero-shot + Code + Context,2741,2,2739,0.07,99.93
9,Llama-4-Maverick,Zero-shot + Code Only,2741,0,2741,0.00,100.00


## Step 12 - Verify Generated Prompt Templates

One representative prompt is generated for each prompt configuration to verify that the prompt builder correctly constructs the expected input before performing the context window analysis.

In [11]:
# ============================================================
# Step 12 - Verify Generated Prompt Templates
# ============================================================

# Use the first sample for verification
sample = working_df.iloc[0]

for config in PROMPT_CONFIGS:

    print("=" * 120)
    print(f"Prompt Configuration : {config['name']}")
    print(f"Strategy             : {config['strategy']}")
    print(f"Include Context      : {config['include_context']}")
    print("=" * 120)

    prompt = build_prompt(
        sample=sample,
        strategy=config["strategy"],
        include_context=config["include_context"]
    )

    token_count = len(tokenizer.encode(prompt))

    print(f"Character Count : {len(prompt):,}")
    print(f"Token Count     : {token_count:,}")
    print()
    print(prompt)
    print("\n\n")

Prompt Configuration : Zero-shot + Code Only
Strategy             : zero_shot
Include Context      : False
Character Count : 2,485
Token Count     : 541

You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-spec